# Fraud Detection Model Training (v2 — real transaction data)

**Purpose.** Train the `RandomForestClassifier` that backs `POST /v2/predict` in the
Enterprise Real-Time Fraud Detection Engine and persist it as
`models/fraud_model_v2.joblib`. The artifact is a dict `{"model": ..., "features": [...]}`
so the serving layer (`app/model_utils.py`) reads the feature order *from the artifact*
and can never silently misalign columns at inference time.

**Dataset.** OpenML data id **45955** (`Credit_Card_Fraud_`, originally published on
Kaggle as `card_transdata.csv`). ~1,000,000 card transactions, ~8.7 % labelled fraud,
licence **Public Domain (CC0)**. It is fetched with `sklearn.datasets.fetch_openml`, so no
Kaggle credentials are needed, and cached under `data/` (git-ignored).

| Feature | Type | Meaning |
|---|---|---|
| `distance_from_home` | float, km | distance between transaction location and cardholder home |
| `distance_from_last_transaction` | float, km | distance from the previous transaction |
| `ratio_to_median_purchase_price` | float | transaction price / cardholder's median purchase price |
| `repeat_retailer` | 0/1 | retailer previously used by this cardholder |
| `used_chip` | 0/1 | EMV chip used |
| `used_pin_number` | 0/1 | PIN entered |
| `online_order` | 0/1 | card-not-present online purchase |
| **`fraud`** | 0/1 | **target** |

**Why replace the synthetic v1 data?** v1 (`fraud_model.joblib`, kept in the repo for the
deprecated `POST /predict`) was trained on `make_classification` output. That proves the
pipeline runs but tells you nothing about fraud. v2 trains on real labelled transactions
with interpretable features, so the metrics below are meaningful — with the caveats spelled
out in the README's *Limitations* section.

**Download budget.** If the OpenML fetch takes longer than five minutes the notebook trains
on a stratified 200,000-row subsample instead of the full million, and records that choice
in the artifact metadata so the README can state it.

In [1]:
import json
import os
import platform
import time
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
OPENML_DATA_ID = 45955
DATA_HOME = os.path.join("..", "data")
ARTIFACT_PATH = os.path.join("..", "models", "fraud_model_v2.joblib")
SAMPLE_CSV_PATH = os.path.join("..", "app", "static", "data", "sample_batch.csv")

DOWNLOAD_BUDGET_SECONDS = 300
SUBSAMPLE_ROWS = 200_000

FEATURES = [
    "distance_from_home",
    "distance_from_last_transaction",
    "ratio_to_median_purchase_price",
    "repeat_retailer",
    "used_chip",
    "used_pin_number",
    "online_order",
]
TARGET = "fraud"
BINARY_FEATURES = ["repeat_retailer", "used_chip", "used_pin_number", "online_order"]

print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| python", platform.python_version())

scikit-learn 1.7.2 | pandas 2.3.3 | python 3.10.11


In [2]:
t0 = time.time()
bunch = fetch_openml(data_id=OPENML_DATA_ID, as_frame=True, data_home=DATA_HOME, parser="auto")
download_seconds = round(time.time() - t0, 1)

full_df = bunch.frame
details = bunch.details or {}
dataset_info = {
    "openml_data_id": OPENML_DATA_ID,
    "openml_name": details.get("name"),
    "openml_version": details.get("version"),
    "licence": details.get("licence"),
    "download_url": details.get("url"),
    "n_rows_available": int(len(full_df)),
    "n_features": len(FEATURES),
    "fraud_count": int(full_df[TARGET].astype(int).sum()),
    "fraud_prevalence": round(float(full_df[TARGET].astype(int).mean()), 5),
    "download_seconds": download_seconds,
    "description": "Synthetic card transactions with engineered behavioural features (Kaggle card_transdata).",
}
print(json.dumps(dataset_info, indent=2))

missing = [c for c in FEATURES + [TARGET] if c not in full_df.columns]
assert not missing, f"dataset is missing expected columns: {missing}"

subsampled = download_seconds > DOWNLOAD_BUDGET_SECONDS
if subsampled:
    df, _ = train_test_split(
        full_df,
        train_size=SUBSAMPLE_ROWS,
        stratify=full_df[TARGET],
        random_state=RANDOM_STATE,
    )
    print(
        f"Download took {download_seconds}s (> {DOWNLOAD_BUDGET_SECONDS}s budget): "
        f"training on a stratified {SUBSAMPLE_ROWS:,}-row subsample."
    )
else:
    df = full_df
    print(f"Download took {download_seconds}s: training on all {len(df):,} rows.")

df = df.copy()
for col in BINARY_FEATURES + [TARGET]:
    df[col] = df[col].astype(int)

print()
print(df[FEATURES].describe().T.round(3))
print()
print("Class balance:")
print(df[TARGET].value_counts(normalize=True).rename("fraction").round(4))
df.head()

{
  "openml_data_id": 45955,
  "openml_name": "Credit_Card_Fraud_",
  "openml_version": "1",
  "licence": "Public Domain (CC0)",
  "download_url": "https://openml.org/data/v1/download/22120398/Credit_Card_Fraud_.arff",
  "n_rows_available": 1000000,
  "n_features": 7,
  "fraud_count": 87403,
  "fraud_prevalence": 0.0874,
  "download_seconds": 1.5,
  "description": "Synthetic card transactions with engineered behavioural features (Kaggle card_transdata)."
}
Download took 1.5s: training on all 1,000,000 rows.



                                    count    mean     std    min    25%  \
distance_from_home              1000000.0  26.629  65.391  0.005  3.878   
distance_from_last_transaction  1000000.0   5.037  25.843  0.000  0.297   
ratio_to_median_purchase_price  1000000.0   1.824   2.800  0.004  0.476   
repeat_retailer                 1000000.0   0.882   0.323  0.000  1.000   
used_chip                       1000000.0   0.350   0.477  0.000  0.000   
used_pin_number                 1000000.0   0.101   0.301  0.000  0.000   
online_order                    1000000.0   0.651   0.477  0.000  0.000   

                                  50%     75%        max  
distance_from_home              9.968  25.744  10632.724  
distance_from_last_transaction  0.999   3.356  11851.105  
ratio_to_median_purchase_price  0.998   2.096    267.803  
repeat_retailer                 1.000   1.000      1.000  
used_chip                       0.000   1.000      1.000  
used_pin_number                 0.000   0.000

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,57.877857,0.311140,1.945940,1,1,0,0,0
1,10.829943,0.175592,1.294219,1,0,0,0,0
2,5.091079,0.805153,0.427715,1,0,0,1,0
3,2.247564,5.600044,0.362663,1,1,0,1,0
4,44.190936,0.566486,2.222767,1,1,0,1,0


## Class imbalance and why PR-AUC, not accuracy

Roughly 8.7 % of rows are fraud. A model that predicts "legitimate" for everything would
score ~91 % accuracy and be useless, so accuracy is not reported as a headline number.

* `class_weight="balanced"` re-weights the loss so each fraud row counts ~10× a legitimate
  row; the forest cannot win by ignoring the minority class.
* **Precision / recall / F1 for the fraud class** describe the operational trade-off:
  precision is the share of flagged transactions that are truly fraudulent (analyst
  workload), recall is the share of fraud that gets caught (loss prevented).
* **PR-AUC (average precision)** summarises that trade-off across every threshold and,
  unlike ROC-AUC, is not inflated by the huge number of easy true negatives.

Nothing below is tuned to flatter the numbers: default `n_estimators`, no depth limit,
no threshold search. The results are reported exactly as they come out.

In [3]:
X = df[FEATURES].to_numpy(dtype=float)
y = df[TARGET].to_numpy(dtype=int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f"train rows: {len(X_train):,} (fraud {y_train.mean():.4%})")
print(f"test  rows: {len(X_test):,} (fraud {y_test.mean():.4%})")

train rows: 800,000 (fraud 8.7402%)
test  rows: 200,000 (fraud 8.7405%)


In [4]:
model = RandomForestClassifier(class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE)

t0 = time.time()
model.fit(X_train, y_train)
fit_seconds = round(time.time() - t0, 1)

n_nodes = int(sum(est.tree_.node_count for est in model.estimators_))
max_depth = int(max(est.tree_.max_depth for est in model.estimators_))
print(f"fit time: {fit_seconds}s | trees: {model.n_estimators} | total nodes: {n_nodes:,} | deepest tree: {max_depth}")

fit time: 39.7s | trees: 100 | total nodes: 43,744 | deepest tree: 33


In [5]:
y_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)

print(classification_report(y_test, y_pred, target_names=["legit", "fraud"], digits=4))

precision, recall, f1, _ = precision_recall_fscore_support(
    y_test, y_pred, average="binary", pos_label=1
)
pr_auc = average_precision_score(y_test, y_proba)
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
tn, fp, fn, tp = (int(v) for v in cm.ravel())

print(f"fraud precision : {precision:.4f}")
print(f"fraud recall    : {recall:.4f}")
print(f"fraud F1        : {f1:.4f}")
print(f"PR-AUC (AP)     : {pr_auc:.4f}")
print()
print("Confusion matrix (rows = actual, cols = predicted):")
print(pd.DataFrame(cm, index=["actual legit", "actual fraud"], columns=["pred legit", "pred fraud"]))
print()

importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("Feature importances (Gini):")
print(importances.round(4).to_string())

metrics = {
    "fraud_precision": round(float(precision), 4),
    "fraud_recall": round(float(recall), 4),
    "fraud_f1": round(float(f1), 4),
    "pr_auc": round(float(pr_auc), 4),
    "confusion_matrix": {"tn": tn, "fp": fp, "fn": fn, "tp": tp},
    "feature_importances": {k: round(float(v), 4) for k, v in importances.items()},
    "n_train_rows": int(len(X_train)),
    "n_test_rows": int(len(X_test)),
    "n_test_fraud": int(y_test.sum()),
    "threshold": 0.5,
}
print()
print("METRICS_JSON", json.dumps(metrics))

              precision    recall  f1-score   support

       legit     1.0000    1.0000    1.0000    182519
       fraud     1.0000    0.9997    0.9999     17481

    accuracy                         1.0000    200000
   macro avg     1.0000    0.9999    0.9999    200000
weighted avg     1.0000    1.0000    1.0000    200000

fraud precision : 1.0000
fraud recall    : 0.9997
fraud F1        : 0.9999
PR-AUC (AP)     : 1.0000

Confusion matrix (rows = actual, cols = predicted):
              pred legit  pred fraud
actual legit      182519           0
actual fraud           5       17476

Feature importances (Gini):
ratio_to_median_purchase_price    0.5416
distance_from_home                0.1986
online_order                      0.1117
distance_from_last_transaction    0.0810
used_pin_number                   0.0348
used_chip                         0.0252
repeat_retailer                   0.0071

METRICS_JSON {"fraud_precision": 1.0, "fraud_recall": 0.9997, "fraud_f1": 0.9999, "pr_auc": 

In [6]:
os.makedirs(os.path.dirname(ARTIFACT_PATH), exist_ok=True)

artifact = {
    "model": model,
    "features": FEATURES,
    "target": TARGET,
    "metrics": metrics,
    "dataset": dataset_info,
    "subsampled": bool(subsampled),
    "n_rows_used": int(len(df)),
    "sklearn_version": sklearn.__version__,
    "trained_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "fit_seconds": fit_seconds,
    "model_params": {"class_weight": "balanced", "n_estimators": model.n_estimators, "random_state": RANDOM_STATE},
}
joblib.dump(artifact, ARTIFACT_PATH, compress=3)
size_mb = os.path.getsize(ARTIFACT_PATH) / 1e6
print(f"Saved {ARTIFACT_PATH} ({size_mb:.2f} MB, compress=3)")

# Round-trip: reload, confirm the feature contract, and score one row of each class.
reloaded = joblib.load(ARTIFACT_PATH)
assert reloaded["features"] == FEATURES, "feature order drifted between save and load"

legit_idx = int(np.flatnonzero(y_test == 0)[0])
fraud_idx = int(np.flatnonzero(y_test == 1)[0])
for label, idx in (("legit", legit_idx), ("fraud", fraud_idx)):
    row = dict(zip(FEATURES, X_test[idx].round(4).tolist()))
    pred = int(reloaded["model"].predict(X_test[idx : idx + 1])[0])
    print(f"{label:>5} example -> predicted {pred}: {row}")

Saved ..\models\fraud_model_v2.joblib (1.14 MB, compress=3)
legit example -> predicted 0: {'distance_from_home': 5.6744, 'distance_from_last_transaction': 0.2321, 'ratio_to_median_purchase_price': 1.1925, 'repeat_retailer': 1.0, 'used_chip': 0.0, 'used_pin_number': 0.0, 'online_order': 1.0}
fraud example -> predicted 1: {'distance_from_home': 198.0127, 'distance_from_last_transaction': 0.1799, 'ratio_to_median_purchase_price': 1.1396, 'repeat_retailer': 1.0, 'used_chip': 0.0, 'used_pin_number': 0.0, 'online_order': 1.0}


## Sample CSV for the batch page

`/batch` accepts a CSV with the seven v2 feature columns. The "Download sample CSV" link on
that page serves a file generated here from **real hold-out rows** (20 legitimate, 5
fraudulent, shuffled) so a visitor can exercise the endpoint with genuine data.

In [7]:
rng = np.random.default_rng(RANDOM_STATE)
legit_rows = rng.choice(np.flatnonzero(y_test == 0), size=20, replace=False)
fraud_rows = rng.choice(np.flatnonzero(y_test == 1), size=5, replace=False)
chosen = np.concatenate([legit_rows, fraud_rows])
rng.shuffle(chosen)

sample = pd.DataFrame(X_test[chosen], columns=FEATURES)
for col in BINARY_FEATURES:
    sample[col] = sample[col].astype(int)
sample[["distance_from_home", "distance_from_last_transaction", "ratio_to_median_purchase_price"]] = (
    sample[["distance_from_home", "distance_from_last_transaction", "ratio_to_median_purchase_price"]].round(4)
)

os.makedirs(os.path.dirname(SAMPLE_CSV_PATH), exist_ok=True)
sample.to_csv(SAMPLE_CSV_PATH, index=False)
print(f"wrote {SAMPLE_CSV_PATH}: {len(sample)} rows ({int(y_test[chosen].sum())} fraud by ground truth)")
sample.head()

wrote ..\app\static\data\sample_batch.csv: 25 rows (5 fraud by ground truth)


,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order
0,40.8014,5.0428,0.0901,1,1,0,0
1,8.8807,0.9113,1.2204,1,1,0,1
2,18.4109,0.7929,0.8198,1,0,0,0
3,6.6014,0.8374,4.7464,1,0,0,1
4,2.1669,1.4352,1.0169,1,1,0,0
